### Import needed libraries

In [6]:
import os
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical # type: ignore
from tensorflow.keras.optimizers import Adam # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import LSTM, Dense, GRU,  Masking, Dropout, Input, BatchNormalization, Layer, GlobalAveragePooling1D, Flatten # type: ignore
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, # type: ignore
                                      TensorBoard, LearningRateScheduler, )

os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/cuda'


In [7]:
X = []
y = []

data_dir = "pickles/"

for file in os.listdir(data_dir):
    file_path = os.path.join(data_dir, file)
    if os.path.exists(file_path):
        with open(file_path, "rb") as f:
            data = pickle.load(f)
        for entry in data:
            points = entry["points"]  # list of frames, each frame = list of (x,y)
            if points:
                # Flatten each frame into 1D vector
                seq = [np.array(frame, dtype=np.float32).flatten() for frame in points]
                X.append(seq)
                y.append(entry["class_name"])
    else:
        print(f"{file} not found!")

print(f"Loaded {len(X)} sequences")

Loaded 1800 sequences


In [8]:
max_seq_len = max(len(seq) for seq in X)
feature_dim = max(len(frame) for seq in X for frame in seq)  # largest frame vector size

X_padded = []
for seq in X:
    arr = np.zeros((max_seq_len, feature_dim), dtype=np.float32)
    for i, frame in enumerate(seq):
        arr[i, :len(frame)] = frame
    X_padded.append(arr)

X_padded = np.array(X_padded, dtype=np.float32)  # (num_samples, max_seq_len, feature_dim)
print("X_padded shape:", X_padded.shape)

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_onehot = to_categorical(y_encoded)

print(X_padded.shape, y_onehot.shape)
print(X_padded[0])
print(X_padded[0][0])

X_padded shape: (1800, 20, 468)
(1800, 20, 468) (1800, 6)
[[225.588   247.67505 227.91295 ...   0.        0.        0.     ]
 [225.7529  248.84538 228.06215 ...   0.        0.        0.     ]
 [226.04338 247.71841 228.32495 ...   0.        0.        0.     ]
 ...
 [226.78261 245.11212 229.22655 ...   0.        0.        0.     ]
 [226.93536 246.35822 229.36322 ...   0.        0.        0.     ]
 [226.69894 245.09451 229.15169 ...   0.        0.        0.     ]]
[225.588   247.67505 227.91295 248.83752 227.91295 248.83752 231.40038
 248.83752 231.40038 248.83752 236.05028 248.83752 236.05028 248.83752
 243.02515 250.      243.02515 250.      250.      250.      250.
 250.      256.97485 250.      256.97485 250.      263.9497  248.83752
 263.9497  248.83752 269.7621  248.83752 269.7621  248.83752 273.24954
 247.67505 273.24954 247.67505 275.5745  247.67505 225.588   247.67505
 226.75047 244.18762 226.75047 244.18762 230.2379  240.7002  230.2379
 240.7002  234.8878  234.8878  234.8878  23

In [17]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import ( # type: ignore
    Input, Masking, Bidirectional, LSTM, LayerNormalization, Dense, Dropout, GlobalAveragePooling1D, MultiHeadAttention
)
from tensorflow.keras.models import Model # type: ignore
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau # type: ignore

# --- Data split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y_onehot, test_size=0.2, random_state=42,
    shuffle=True, stratify=y_encoded
)

inputs = Input(shape=(20, 468))

x = Bidirectional(LSTM(128, return_sequences=True))(inputs)
x = Bidirectional(LSTM(128, return_sequences=True))(x)

# Multi-head self-attention
attn = MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
x = LayerNormalization()(x + attn)  # residual connection + layer norm

x = GlobalAveragePooling1D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.7)(x)
x = Dense(128, activation="relu")(x)
outputs = Dense(y_onehot.shape[1], activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Callbacks
early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5)

history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr]
)

loss, acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {acc:.3f}")



Epoch 1/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.4701 - loss: 1.3991 - val_accuracy: 0.8000 - val_loss: 0.6737 - learning_rate: 1.0000e-04
Epoch 2/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7507 - loss: 0.7562 - val_accuracy: 0.9028 - val_loss: 0.4171 - learning_rate: 1.0000e-04
Epoch 3/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8007 - loss: 0.5734 - val_accuracy: 0.8889 - val_loss: 0.3531 - learning_rate: 1.0000e-04
Epoch 4/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8528 - loss: 0.4436 - val_accuracy: 0.8750 - val_loss: 0.3246 - learning_rate: 1.0000e-04
Epoch 5/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8701 - loss: 0.4122 - val_accuracy: 0.9083 - val_loss: 0.2647 - learning_rate: 1.0000e-04
Epoch 6/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8687 - loss: 0.3805 - val_accuracy: 0.9417 - val_loss: 0.2164 - learning_rate: 1.0000e-04
Epoch 7/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 

In [10]:
model.save("gesture_model.keras")

In [11]:
# import libraries
with open('label_encoder.pkl', 'wb') as f:
  pickle.dump(le, f)

In [14]:
import tensorflow as tf
print("TF:", tf.__version__)
print("Physical GPUs:", tf.config.list_physical_devices("GPU"))
try:
    from tensorflow.python.client import device_lib
    print(device_lib.list_local_devices())
except Exception:
    pass


TF: 2.20.0
Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 15316918136786916390
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 5420875776
locality {
  bus_id: 1
  links {
  }
}
incarnation: 9392287807250820156
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6"
xla_global_id: 416903419
]


I0000 00:00:1758572161.790073  437593 gpu_device.cc:2020] Created device /device:GPU:0 with 5169 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6


In [15]:
import numpy as np
np.sum(y_train, axis=0)  # class counts


array([60., 60., 60., 60., 60., 60.])